# VINGAGE – Notebook 2: SHAP and GRU on a Colab GPU
This notebook does **not need the DAiSEE videos**. It only reads the feature table and the per-frame sequences that notebook 1 saved to your Drive (`daisee_project/features_all.csv` and `daisee_project/chunks/seq/`).

Before running: **Runtime → Change runtime type → T4 GPU**, then run the cells in order. (Changing the runtime type restarts the machine, which is fine because everything needed is on Drive.)

In [ ]:
!nvidia-smi | head -15
import torch; print('GPU available:', torch.cuda.is_available())

In [ ]:
!pip -q install shap xgboost
from google.colab import drive
drive.mount('/content/drive')
import os; WORK = '/content/drive/MyDrive/daisee_project'
print(os.path.exists(f'{WORK}/features_all.csv'), len(os.listdir(f'{WORK}/chunks/seq')), 'sequence files')

In [ ]:
%%writefile /content/ed_models.py
"""Baselines on the clip feature table. Reports accuracy, macro-F1 and per-class recall."""
import numpy as np, pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, f1_score, recall_score
from sklearn.utils.class_weight import compute_sample_weight

NON_FEATURE = {"clip", "split", "subject", "Boredom", "Engagement", "Confusion", "Frustration", "error", "secs", "clahe_frames"}

def feature_cols(F):
    return [c for c in F.columns if c not in NON_FEATURE and pd.api.types.is_numeric_dtype(F[c])]

def make_xy(F, task):
    y = F["Engagement"].astype(int)
    if task == "binary":
        y = (y >= 2).astype(int)
    return y

def run_baselines(F, task="four", min_detect=0.0, seed=0):
    from xgboost import XGBClassifier
    F = F[(F["error"] == "") & (F["detect_rate"] >= min_detect)]
    cols = feature_cols(F)
    tr, va, te = (F[F.split == s] for s in ("Train", "Validation", "Test"))
    med = tr[cols].median()
    X = {k: d[cols].fillna(med).fillna(0).values for k, d in (("val", va), ("test", te), ("train", tr))}
    ytr = make_xy(tr, task).values; classes = sorted(set(ytr)); remap = {c: i for i, c in enumerate(classes)}
    labels = [0, 1] if task == "binary" else [0, 1, 2, 3]
    models = {
        "majority": DummyClassifier(strategy="most_frequent"),
        "logistic_regression": make_pipeline(StandardScaler(), LogisticRegression(class_weight="balanced", max_iter=3000)),
        "xgboost": XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
                                 random_state=seed, n_jobs=2, eval_metric="mlogloss"),
    }
    out, fitted = [], {}
    for name, m in models.items():
        yt = np.array([remap[v] for v in ytr])
        if name == "xgboost":
            m.fit(X["train"], yt, sample_weight=compute_sample_weight("balanced", yt))
        else:
            m.fit(X["train"], yt)
        fitted[name] = m
        for split, d in (("Validation", va), ("Test", te)):
            y = make_xy(d, task).values
            p = np.array([classes[i] for i in m.predict(X["val" if split == "Validation" else "test"])])
            row = {"model": name, "split": split, "n": len(y), "accuracy": accuracy_score(y, p),
                   "macro_F1": f1_score(y, p, labels=labels, average="macro", zero_division=0)}
            for l, r in zip(labels, recall_score(y, p, labels=labels, average=None, zero_division=0)):
                row[f"recall_{l}"] = r
            out.append(row)
    return pd.DataFrame(out).round(3), fitted, cols


In [ ]:
%%writefile /content/ed_gru.py
"""GRU on per-frame signal sequences (run on a Colab GPU). Same splits and metrics as the baselines."""
import os, numpy as np, pandas as pd, torch, torch.nn as nn
from sklearn.metrics import accuracy_score, f1_score, recall_score
from sklearn.utils.class_weight import compute_class_weight

T = 50

def load_split(F, seq_dir, split, task):
    d = F[(F.split == split) & (F.error == "")]
    X, y, keep = [], [], []
    for clip, e in zip(d["clip"], d["Engagement"].astype(int)):
        p = os.path.join(seq_dir, f"{clip}.npy")
        if not os.path.exists(p): continue
        a = np.load(p)[:T]
        if len(a) < T: a = np.vstack([a, np.full((T - len(a), a.shape[1]), np.nan, np.float32)])
        X.append(a); y.append(int(e >= 2) if task == "binary" else e); keep.append(clip)
    return np.stack(X), np.array(y), keep

def prep(X, mu=None, sd=None):
    """Standardise signals; every NaN becomes 0 with a per-signal 'valid' flag so no NaN reaches the network."""
    sig = X[..., :-1]
    if mu is None:
        flat = sig.reshape(-1, sig.shape[-1])
        mu = np.nan_to_num(np.nanmean(flat, axis=0)); sd = np.nan_to_num(np.nanstd(flat, axis=0), nan=1.0) + 1e-6
    Z = (sig - mu) / sd
    valid = ~np.isnan(Z)
    Z = np.clip(np.where(valid, Z, 0.0), -10, 10)
    det = np.nan_to_num(X[..., -1:], nan=0.0)
    out = np.concatenate([Z, valid.astype(np.float32), det], axis=-1).astype(np.float32)
    assert np.isfinite(out).all(), "non-finite values reached the GRU input"
    return out, mu, sd

class GRUNet(nn.Module):
    def __init__(self, d_in, n_cls, h=64):
        super().__init__(); self.gru = nn.GRU(d_in, h, batch_first=True); self.drop = nn.Dropout(0.3); self.fc = nn.Linear(h, n_cls)
    def forward(self, x):
        _, hN = self.gru(x); return self.fc(self.drop(hN[-1]))

def focal(logits, y, w, gamma=2.0):
    lp = torch.log_softmax(logits, 1); p = lp.exp()
    return (-(w[y] * (1 - p.gather(1, y[:, None])[:, 0]) ** gamma * lp.gather(1, y[:, None])[:, 0])).mean()

def metrics(y, p, labels):
    r = {"accuracy": accuracy_score(y, p), "macro_F1": f1_score(y, p, labels=labels, average="macro", zero_division=0)}
    for l, v in zip(labels, recall_score(y, p, labels=labels, average=None, zero_division=0)): r[f"recall_{l}"] = v
    return r

def train_gru(F, seq_dir, task="four", epochs=40, seed=0, patience=8):
    torch.manual_seed(seed); np.random.seed(seed)
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    Xtr, ytr, _ = load_split(F, seq_dir, "Train", task); Xva, yva, _ = load_split(F, seq_dir, "Validation", task); Xte, yte, _ = load_split(F, seq_dir, "Test", task)
    Xtr, mu, sd = prep(Xtr); Xva, _, _ = prep(Xva, mu, sd); Xte, _, _ = prep(Xte, mu, sd)
    labels = [0, 1] if task == "binary" else [0, 1, 2, 3]
    cw = np.ones(len(labels)); present = np.unique(ytr)
    cw[present] = compute_class_weight("balanced", classes=present, y=ytr)
    w = torch.tensor(cw, dtype=torch.float32, device=dev)
    net = GRUNet(Xtr.shape[-1], len(labels)).to(dev); opt = torch.optim.Adam(net.parameters(), 1e-3, weight_decay=1e-4)
    tt = lambda a: torch.tensor(a, device=dev)
    xtr, ytr_t = tt(Xtr), tt(ytr).long()
    best, best_state, bad = -1, None, 0
    for ep in range(epochs):
        net.train(); perm = torch.randperm(len(xtr), device=dev)
        for i in range(0, len(perm), 64):
            idx = perm[i:i + 64]; opt.zero_grad(); loss = focal(net(xtr[idx]), ytr_t[idx], w)
            if not torch.isfinite(loss): raise RuntimeError('loss became NaN/inf')
            loss.backward(); torch.nn.utils.clip_grad_norm_(net.parameters(), 1.0); opt.step()
        net.eval()
        with torch.no_grad(): pv = net(tt(Xva)).argmax(1).cpu().numpy()
        f1 = f1_score(yva, pv, labels=labels, average="macro", zero_division=0)
        if f1 > best: best, bad, best_state = f1, 0, {k: v.clone() for k, v in net.state_dict().items()}
        else:
            bad += 1
            if bad >= patience: break
    net.load_state_dict(best_state); net.eval(); out = []
    with torch.no_grad():
        for name, X, y in (("Validation", Xva, yva), ("Test", Xte, yte)):
            p = net(tt(X)).argmax(1).cpu().numpy(); print(name, "predicted class counts:", dict(zip(*np.unique(p, return_counts=True)))); out.append({"model": "gru", "split": name, "n": len(y), **metrics(y, p, labels)})
    return pd.DataFrame(out).round(3), net


## Load the saved features

In [ ]:
import sys; sys.path.insert(0, '/content')
import pandas as pd, numpy as np
F = pd.read_csv(f'{WORK}/features_all.csv', dtype={'clip': str}); F['error'] = F['error'].fillna('')
print(F.shape); print(F.groupby('split').agg(clips=('clip', 'count'), face_detect_rate=('detect_rate', 'mean')).round(3))
print('clips with an error:', (F.error != '').sum(), '| clips with no face at all:', (F.detect_rate == 0).sum())
print(F.groupby('split').Engagement.value_counts(normalize=True).unstack().round(3))

## SHAP on XGBoost (which signals drive the four-level prediction)

In [ ]:
import ed_models as M, shap, matplotlib.pyplot as plt
r4, fit4, cols = M.run_baselines(F, task='four')
tr = F[(F.split == 'Train') & (F.error == '')]; med = tr[cols].median()
Xte = F[(F.split == 'Test') & (F.error == '')][cols].fillna(med).fillna(0)
sv = shap.TreeExplainer(fit4['xgboost']).shap_values(Xte)
shap.summary_plot(sv, Xte, plot_type='bar', max_display=15, show=False)
plt.tight_layout(); plt.savefig(f'{WORK}/shap_bar_four.png', dpi=200); plt.show()

## GRU on the per-frame sequences (GPU)

In [ ]:
import ed_gru as G
res = []
for task in ('four', 'binary'):
    r, net = G.train_gru(F, f'{WORK}/chunks/seq', task=task)
    r['task'] = task; res.append(r); print(task); print(r.to_string(index=False))
pd.concat(res).to_csv(f'{WORK}/gru_results.csv', index=False)

## Next
Send me the printed tables and the SHAP plot. Real numbers go into Table 7 of the report only from this output. After this: the darkened-input test (with and without CLAHE) and the live demo.